# Offline activity module III

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats

### Example

In [29]:
# Let's say we have bacteria growing on different carbon sources and we measure the doubling time of
# each colony grown on an agar plate of one condition and another. This will be our example data for
# independent groups. The doubling time is given in h.
glucose = np.random.uniform(0.25,0.6,30)
tryptone = np.random.uniform(0.45,0.8,30)

# For dependent groups, let's pretend we ran killing curve experiments on our bacteria testing
# antibiotic killing of persistent bacteria over time. For some inexplicable scientifically
# disappointing reason, however, we only performed CFU counts of our replicates at timepoint
# zero and 24 hours after addition of the antibiotic ciprofloxacin. This will be our second
# example dataset.
t0 = np.random.uniform(0.78,0.935,50)
t24 = np.random.uniform(0.113,0.381,50)

# To also include negative data for the dependent dataset, let's look at the relative expression
# levels of genes in E. coli conferring resistance during and after high antibiotic (nitrofurantoin) 
# stress (compared to before). The data is recorded 4 hours into the treatment and 48 hours after the
# treatment was stopped.
stress = np.random.uniform(-0.1,3,40)
recovery = np.random.uniform(-1,1.85,40)

### Student's t-test (two-sample t-test)

The two-sample t-test predicts y with the model $a + b * x_i$ where $x_i$ is 0 for the first group and 1 for the second. It tests the null hypothesis that randomly selected values of two datasets have the same distribution. This is a parametric test using the actual data and assumes normally distributed data. The paired test also assumes the two datasets are related while the independent test, evidently, treats them as independent.

In [36]:
def ttest(df1,df2, method):
    if method == 'independent':
        # calculate the means and pooled sample variance
        m1, m2 = np.mean(df1), np.mean(df2)
        spsq = ((len(df1)-1)*np.var(df1,ddof=1) + (len(df2)-1)*np.var(df2,ddof=1))/(len(df1) + len(df2) - 2)
    
        # calculate the t statistic
        t = (m1 - m2)/np.sqrt((spsq/len(df1))+(spsq/len(df2)))
    
    elif method == 'paired':
        # calculate the mean of the differences between the two datasets and the standard deviation
        dif = []
        for x in range(len(df1)):
            dif.append(df1[x]-df2[x])
        difmean = np.mean(dif)
        difsd = np.std(dif,ddof=1)
        
        # calculate the t statistic
        t = difmean / (difsd/np.sqrt(len(df1)))
    
    return t
ind_t = ttest(glucose,tryptone, method='independent')
pa_t = ttest(t0,t24, method='paired')
print(f'the computed independent t-statistic for growth on different carbon sources is: {ind_t},' + '\n' +
      f'and the paired t-statistic of the killing curve is: {pa_t}.')

the computed independent t-statistic for growth on different carbon sources is: -6.396918366423253,
and the paired t-statistic of the killing curve is: 50.452201123731555.


### Mann-Whitney test

The Mann-Whitney U test is the non-parametric alternative to the independent t-test using the ranks of the x and y data respectively. This means that the data does not have to be normally distributed. The test tests the same null hypothesis as the independent t-test. In this case, the model is based on $ranked(y_i) = a + b*x_i$. $x_i$, again, is 0 for the first group and 1 for the second.

In [35]:
def MWUtest(df1,df2):
    # rank the data and calculate the rank sums
    n1, n2 = len(df1), len(df2)
    combined = np.concatenate([df1,df2])
    ranks = stats.rankdata(combined)
    r1,r2 = np.sum(ranks[:n1]), np.sum(ranks[n1:])
    
    # calculate the U statistic
    u1 = n1*n2 + (n1*(n1+1))/2 - r1
    u2 = n1*n2 + (n2*(n2+1))/2 - r2
    
    return min([u1,u2])

ustat = MWUtest(glucose,tryptone)
print(f'the computed U-statistic for growth on different carbon sources is: {ustat}.')

the computed U-statistic for growth on different carbon sources is: 125.0.


### Wilcoxon signed-rank test

Similar to the Mann-Whitney U-test, the Wilcoxon signed-rank test is a non-parametric alternative statistical test to the paired two-sample t-test. Thus, by being non-parametric the data does not have to meet the assumption that it is normally distributed. The test evaluates the difference between two dependent groups of data. It uses signed ranks (ranked data to which the signs of the data are added).

In [34]:
def wilcox(df1,df2):
    # calculate paired differences
    dif = []
    for x in range(len(df1)):
        dif.append(df1[x]-df2[x])
    absdif = []
    for n in dif:
        if n < 0:
            absdif.append(-n)
        else:
            absdif.append(n)

    # Rank the data
    ranks = stats.rankdata(absdif)
    
    # Calculate the w statistic
    tpos, tneg = 0, 0
    for n in range(len(dif)):
        if dif[n] < 0:
            tneg += ranks[n]
        else:
            tpos += ranks[n]
    
    return min([tpos,tneg])

wstat = wilcox(stress,recovery)
print(f'the computed W-statistic for differential gene expression in antibiotic stress conditions is: {wstat}.')

the computed W-statistic for differential gene expression in antibiotic stress conditions is: 101.0.


### Wald test
A Wald test is a statistical test to evaluate whether a given statistic (e.g., mean) is significantly different to some value (usually 0) given the uncertainty of the data (e.g., standard deviation). Essentially, it tests the significance of a signal in relation to the noise. An independent t-test can be seen as a special case of a Wald test and is used for smaller samples with defined distributions (normal), while the Wald test can be used for large samples.

In [37]:
def wald(df,Tnull,T=None, var=None): # can use the function with vectors of parameters.
    # Tnull are the predicted parameters of the null hypothesis.
    if T == None:
        # let's use a mean and standard error as an estimate and variance
        estimate = np.mean(df)
        var = np.var(df,ddof=1)
        
        # calculate the W statistic
        W = ((estimate-Tnull)**2)/ var
    else:
        W = (np.array(T) - np.array(Tnull))**2 / var
    
    return W
Wstat = wald(stress,0)
print(f'the computed Wald-statistic for differential gene expression in antibiotic stress conditions is: {Wstat}.')

the computed Wald-statistic for differential gene expression in antibiotic stress conditions is: 2.7298024128174725.


### Conclusion
In the end, all of these statistical tests evaluate whether one or two samples differ significantly from each other or from a defined null hypothesis value. Parametric tests (t-test) assume certain things about the data like the distribution and use the data itself. They tend to be more accurate (t-test is also more accurate for small samples) but are limited by their assumptions. Non-parametric tests are not limited by these assumptions and can deal with skewed data but are less accurate. But the literature you gave us to read made it very clear; despite everything all of these tests at the end of the day rely on simple linear models for prediction.